# Inter-event time (`dt`) vs time

Load the latest swaps file as an `EventCatalogs` bundle, then plot
`dt` between successive events against event time for snaps, cascades,
and avalanches.

Times are shifted by a catalog origin (`t − t₀`) so the x-axis begins near 0.
Plots are interactive Plotly figures (log–log).

The same plot is repeated for the JMA and Hauksson earthquake catalogs
(`data/ingested/jma.csv`, `data/ingested/hauksson.csv`):

- **JMA** — years 2009–2013, with `t₀` just before the Tohoku-oki mainshock
- **Hauksson** — years 2017–2021, with `t₀` just before the Ridgecrest mainshock


In [1]:
from __future__ import annotations

import pathlib
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "pyproject.toml").is_file() and (base / "crumpling").is_dir():
            return base
    raise FileNotFoundError(f"Could not locate the repository above {here}")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from crumpling import catalogs
from crumpling import config
from crumpling import pipeline


## 1. Latest simulation → `EventCatalogs`

Pick the most recently modified `*__swaps.txt` under `data/raw/`, run the
pipeline, and wrap the result with `build_catalogs`.

In [2]:
swap_candidates = sorted(
    (ROOT / "data/raw").glob("**/N*__swaps*.txt"),
    key=lambda path: path.stat().st_mtime,
)
if not swap_candidates:
    raise FileNotFoundError("No swaps files found under data/raw")

DATA = swap_candidates[-1]
CONFIG = ROOT / "configs/trial32/swapstat.params.ini"

settings = config.load_config(CONFIG)
result = pipeline.analyze_file(DATA, settings)
bundle = catalogs.build_catalogs(result)

print(type(bundle).__name__)
print("source:", bundle.source)
print(
    f"snaps={len(bundle.snaps)}  "
    f"cascades={len(bundle.cascades)}  "
    f"avalanches={len(bundle.avalanches)}"
)


EventCatalogs
source: /home/neriberman/REPOS/crumpling/data/raw/trial32/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt
snaps=3242  cascades=892  avalanches=225


## 2. Simulation hierarchies: `dt` vs time

Each catalog already has a `dt` column (`time.diff()` after time-sorting).
X is relative time `t − t₀` (first event at 0). Log–log; the first event
(`dt` NaN) and non-positive relative times are omitted.

In [3]:
def dt_vs_relative_time(
    frame: pd.DataFrame,
    *,
    t0: float | None = None,
    max_points: int | None = None,
) -> pd.DataFrame:
    """Return columns ``t`` (time − t₀) and ``dt`` for plotting.

    If ``t0`` is omitted, the first event time is used. Callers that want a
    mainshock at a small positive ``t`` (for log axes) should pass
    ``t0 = t_main - eps``.
    """

    series = frame.loc[:, ["time", "dt"]].dropna(subset=["time"]).copy()
    if t0 is None:
        t0 = float(series["time"].iloc[0]) if len(series) else 0.0
    series["t"] = series["time"] - float(t0)
    series = series.loc[(series["t"] > 0) & (series["dt"] > 0), ["t", "dt"]]
    if max_points is not None and len(series) > max_points:
        step = int(np.ceil(len(series) / max_points))
        series = series.iloc[::step]
    return series


def plot_dt_vs_time(
    frame: pd.DataFrame,
    title: str,
    *,
    t0: float | None = None,
    time_label: str = "t − t₀",
    dt_label: str = "dt",
    max_points: int | None = None,
    height: int = 420,
) -> go.Figure:
    series = dt_vs_relative_time(frame, t0=t0, max_points=max_points)
    fig = go.Figure(
        data=[
            go.Scattergl(
                x=series["t"],
                y=series["dt"],
                mode="markers",
                marker=dict(size=5, opacity=0.45),
                name=title,
            )
        ]
    )
    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=60, r=20, t=50, b=50),
        xaxis=dict(title=time_label, type="log"),
        yaxis=dict(title=dt_label, type="log"),
    )
    return fig


hierarchies = (
    ("snaps", bundle.snaps),
    ("cascades", bundle.cascades),
    ("avalanches", bundle.avalanches),
)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[name for name, _ in hierarchies],
)
for row, (name, frame) in enumerate(hierarchies, start=1):
    series = dt_vs_relative_time(frame)
    fig.add_trace(
        go.Scattergl(
            x=series["t"],
            y=series["dt"],
            mode="markers",
            marker=dict(size=5, opacity=0.45),
            name=name,
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="dt", type="log", row=row, col=1)

fig.update_xaxes(title_text="t − t₀", type="log", row=3, col=1)
fig.update_xaxes(type="log", row=1, col=1)
fig.update_xaxes(type="log", row=2, col=1)
fig.update_layout(
    title_text=f"dt vs time (log–log) — {bundle.source.name}",
    height=900,
    margin=dict(l=60, r=20, t=80, b=50),
)
fig.show()


## 3. JMA catalog (2009–2013)

Filter `data/ingested/jma.csv` to Unix times in `[2009-01-01, 2014-01-01)`.
Set `t₀` to just before the Tohoku-oki mainshock (`M 9.0`, 2011-03-11) so that
event sits at a small positive `t` on the log–x axis (`t₀ = t_main − ε`).


In [4]:
JMA_PATH = ROOT / "data/ingested/jma.csv"
T0_EPS = 1.0  # seconds: mainshock appears at t = ε > 0 on log axes

jma_start = pd.Timestamp("2009-01-01").timestamp()
jma_end = pd.Timestamp("2014-01-01").timestamp()

jma = pd.read_csv(JMA_PATH, usecols=["time", "magnitude"])
jma = jma.loc[(jma["time"] >= jma_start) & (jma["time"] < jma_end)].copy()
jma = jma.sort_values("time", kind="mergesort").reset_index(drop=True)
jma["dt"] = jma["time"].diff()
jma["datetime"] = pd.to_datetime(jma["time"], unit="s")

# Tohoku-oki: largest event on 2011-03-11 (M 9.0).
tohoku_day0 = pd.Timestamp("2011-03-11").timestamp()
tohoku_day1 = pd.Timestamp("2011-03-12").timestamp()
tohoku = (
    jma.loc[(jma["time"] >= tohoku_day0) & (jma["time"] < tohoku_day1)]
    .sort_values("magnitude", ascending=False)
    .iloc[0]
)
jma_t0 = float(tohoku["time"]) - T0_EPS

print(
    f"JMA 2009–2013: {len(jma):,} events  "
    f"({jma['datetime'].iloc[0]} → {jma['datetime'].iloc[-1]})"
)
print(f"median dt = {jma['dt'].median():.3f} s")
print(
    f"Tohoku-oki: M={tohoku['magnitude']:.1f} at {tohoku['datetime']}  "
    f"(Unix {float(tohoku['time']):.3f})"
)
print(f"t₀ = Tohoku − {T0_EPS:g} s → {pd.to_datetime(jma_t0, unit='s')}")


JMA 2009–2013: 888,516 events  (2009-01-01 00:02:25.539999962 → 2013-12-31 23:56:22.140000105)
median dt = 98.950 s
Tohoku-oki: M=9.0 at 2011-03-11 05:46:18.119999886  (Unix 1299822378.120)
t₀ = Tohoku − 1 s → 2011-03-11 05:46:17.119999886


In [5]:
fig = plot_dt_vs_time(
    jma,
    "JMA 2009–2013 (t₀ = Tohoku − ε)",
    t0=jma_t0,
    time_label="t − t₀ (s)",
    dt_label="dt (s)",
    max_points=80_000,
    height=480,
)
fig.show()


## 4. Hauksson catalog (2017–2021)

Filter `data/ingested/hauksson.csv` to Unix times in `[2017-01-01, 2022-01-01)`
so the window contains Ridgecrest. Set `t₀` to just before the Ridgecrest
mainshock (`M 7.1`, 2019-07-06): `t₀ = t_main − ε`.


In [6]:
HAUKSSON_PATH = ROOT / "data/ingested/hauksson.csv"

hauk_start = pd.Timestamp("2017-01-01").timestamp()
hauk_end = pd.Timestamp("2022-01-01").timestamp()

hauksson = pd.read_csv(HAUKSSON_PATH, usecols=["time", "magnitude"])
hauksson = hauksson.loc[
    (hauksson["time"] >= hauk_start) & (hauksson["time"] < hauk_end)
].copy()
hauksson = hauksson.sort_values("time", kind="mergesort").reset_index(drop=True)
hauksson["dt"] = hauksson["time"].diff()
hauksson["datetime"] = pd.to_datetime(hauksson["time"], unit="s")

# Ridgecrest sequence: take the M 7.1 on 2019-07-06 as the mainshock.
ridge_day0 = pd.Timestamp("2019-07-06").timestamp()
ridge_day1 = pd.Timestamp("2019-07-07").timestamp()
ridgecrest = (
    hauksson.loc[
        (hauksson["time"] >= ridge_day0) & (hauksson["time"] < ridge_day1)
    ]
    .sort_values("magnitude", ascending=False)
    .iloc[0]
)
hauksson_t0 = float(ridgecrest["time"]) - T0_EPS

print(
    f"Hauksson 2017–2021: {len(hauksson):,} events  "
    f"({hauksson['datetime'].iloc[0]} → {hauksson['datetime'].iloc[-1]})"
)
print(f"median dt = {hauksson['dt'].median():.3f} s")
print(
    f"Ridgecrest: M={ridgecrest['magnitude']:.2f} at {ridgecrest['datetime']}  "
    f"(Unix {float(ridgecrest['time']):.3f})"
)
print(
    f"t₀ = Ridgecrest − {T0_EPS:g} s → "
    f"{pd.to_datetime(hauksson_t0, unit='s')}"
)


Hauksson 2017–2021: 60,943 events  (2017-01-01 00:04:55 → 2021-12-31 23:38:36)
median dt = 866.000 s
Ridgecrest: M=7.10 at 2019-07-06 03:19:52  (Unix 1562383192.000)
t₀ = Ridgecrest − 1 s → 2019-07-06 03:19:51


In [7]:
fig = plot_dt_vs_time(
    hauksson,
    "Hauksson 2017–2021 (t₀ = Ridgecrest − ε)",
    t0=hauksson_t0,
    time_label="t − t₀ (s)",
    dt_label="dt (s)",
    max_points=80_000,
    height=480,
)
fig.show()
